# Entrenamiento de detector de cabezas con YOLO (Ultralytics)

Este notebook hace **fine-tuning** de un detector de cabezas sobre el dataset **CrowdHuman** ya preparado en `data/crowdhuman/yolo/`.

Como checkpoint de partida se usa el modelo descargado de Kaggle:
`~/Downloads/human-head-detection-pytorch-yolov5mu-v1.tar.gz` (contiene `yolov5mu-finetuned.pt`).

**Dataset:** 15.000 imágenes train / 4.370 val, una sola clase (`head`).

**Hardware detectado:** NVIDIA RTX 4060 Ti (16 GB).

## 1. Dependencias
Descomenta la celda si Ultralytics no está instalado en tu entorno.

In [ ]:
# %pip install -U ultralytics

## 2. Configuración de rutas

In [ ]:
from pathlib import Path

# Raiz del proyecto (el notebook vive en notebooks/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data" / "crowdhuman" / "yolo"
MODEL_TAR = Path.home() / "Downloads" / "human-head-detection-pytorch-yolov5mu-v1.tar.gz"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "head_detector"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:    ", DATA_DIR, "->", "OK" if DATA_DIR.exists() else "NO EXISTE")
print("MODEL_TAR:   ", MODEL_TAR, "->", "OK" if MODEL_TAR.exists() else "NO EXISTE")

## 3. Extraer el checkpoint base del tar.gz
El `.tar.gz` contiene `yolov5mu-finetuned.pt`. Lo extraemos a `models/`.

In [ ]:
import tarfile

BASE_CKPT = MODELS_DIR / "yolov5mu-finetuned.pt"

if not BASE_CKPT.exists():
    with tarfile.open(MODEL_TAR, "r:gz") as tar:
        members = tar.getnames()
        print("Contenido del tar:", members)
        pt_name = next(m for m in members if m.endswith(".pt"))
        tar.extract(pt_name, path=MODELS_DIR)
        extracted = MODELS_DIR / pt_name
        if extracted != BASE_CKPT:
            extracted.rename(BASE_CKPT)

print("Checkpoint base:", BASE_CKPT, "->", "OK" if BASE_CKPT.exists() else "FALLO")

## 4. Crear el `data.yaml` con rutas locales
El `crowdhuman_head.yaml` del repo apunta a `/workspace/...` (rutas del contenedor). Generamos uno con rutas locales absolutas para entrenar fuera de Docker.

In [ ]:
import yaml

data_cfg = {
    "path": str(DATA_DIR),
    "train": "images/train",
    "val": "images/val",
    "names": {0: "head"},
}

DATA_YAML = PROJECT_ROOT / "data" / "crowdhuman_head_local.yaml"
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

print("data.yaml ->", DATA_YAML)
print(DATA_YAML.read_text())

n_train = len(list((DATA_DIR / "images" / "train").glob("*.jpg")))
n_val = len(list((DATA_DIR / "images" / "val").glob("*.jpg")))
print(f"train: {n_train} imgs | val: {n_val} imgs")

## 5. Comprobar GPU

In [ ]:
import torch

print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM total (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
DEVICE = 0 if torch.cuda.is_available() else "cpu"

## 6. Entrenamiento

Cargamos el checkpoint base (`yolov5mu-finetuned.pt`) y entrenamos sobre CrowdHuman.

- `batch=0.85` deja que Ultralytics ajuste el batch al 85% de la VRAM (≈14 GB de los 16 GB).
- `cache="disk"` evita saturar la RAM con 15k imágenes.
- Ajusta `epochs` según el tiempo disponible (50 es un buen punto de partida).

In [ ]:
from ultralytics import YOLO

model = YOLO(str(BASE_CKPT))

results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=0.85,
    cache="disk",
    workers=8,
    device=DEVICE,
    project=str(OUTPUT_DIR),
    name="yolov5mu-crowdhuman-head",
    patience=15,
    plots=True,
)

## 7. Validación
Métricas (mAP) sobre el split de validación con los mejores pesos.

In [ ]:
best_weights = Path(results.save_dir) / "weights" / "best.pt"
print("Mejores pesos:", best_weights)

best = YOLO(str(best_weights))
metrics = best.val(data=str(DATA_YAML), imgsz=640, device=DEVICE)
print("mAP50-95:", metrics.box.map)
print("mAP50:   ", metrics.box.map50)

## 8. Inferencia de prueba
Predecimos sobre unas imágenes de validación y mostramos los resultados.

In [ ]:
import matplotlib.pyplot as plt

sample_imgs = sorted((DATA_DIR / "images" / "val").glob("*.jpg"))[:4]
preds = best.predict(source=[str(p) for p in sample_imgs], imgsz=640, conf=0.25, device=DEVICE)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, pred in zip(axes.ravel(), preds):
    ax.imshow(pred.plot()[:, :, ::-1])  # BGR -> RGB
    ax.set_title(f"{len(pred.boxes)} cabezas")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 9. (Opcional) Exportar el modelo
Exporta a ONNX para despliegue / inferencia optimizada.

In [ ]:
# best.export(format="onnx", imgsz=640, dynamic=True)